# Rest / movement classification (`pirouette_data.behavior_classification`)

This notebook walks through the velocity-based behaviour classifier:

1. Compute the **smoothed** ear-midpoint velocity (classification uses this to avoid instantaneous spikes).
2. Pick a speed threshold with **Otsu** (log-scale, to capture slow movement).
3. Enforce a **minimum bout duration** (0.5 s) so only sustained movement counts.
4. Append a `behavior` label column and inspect the result (ethogram, bout durations).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pirouette_data import (
    behavior_classification as bc,
    ingestion,
    kinematics,
    processing,
)

pd.set_option("display.max_columns", 60)

## Build the inputs

Load one hour of pose data, attach real Harp time, convert to mm, and compute the instantaneous
**and** smoothed ear-midpoint velocity.

In [ ]:
POSE_DIR = r"C:/Users/brandon.pratt/Desktop/data/body-kinematics/pose_data"
S3_VIDEO_URI = "s3://aind-open-data/854393_2026-06-09_19-34-26/behavior-videos"
LIKELIHOOD = 0.6

fname = sorted(Path(POSE_DIR).glob("*.h5"))[0]
df = ingestion.load_pose_h5(fname).reset_index()
camera, timestamp, _ = ingestion.parse_camera_and_timestamp(fname.name)
df["harp_time"] = ingestion.load_harp_seconds(S3_VIDEO_URI, camera, timestamp)[: len(df)]
df = processing.append_mm_columns(df, likelihood_threshold=LIKELIHOOD)
df = kinematics.append_ear_velocity(df, likelihood_threshold=LIKELIHOOD, smoothing_sigma=1.5)

fps = 1.0 / np.median(np.diff(df["harp_time"]))
print(f"{fname.name}: {df.shape}   fps ~ {fps:.1f}")

## Why classify on the *smoothed* velocity?

The instantaneous velocity is spiky; thresholding it produces many spurious short bouts. The
classifier defaults to `ear_velocity_smooth_mm_s` (Gaussian-smoothed) to remove that noise **at the
source**. Below we disable the min-bout filter to see the raw effect of the spikes.

In [ ]:
def n_movement_bouts(labels):
    b = (np.asarray(labels) == "movement").astype(int)
    return int((np.diff(b) == 1).sum() + (b[0] == 1))


# Disable the min-bout filter so the raw spike-driven bouts are visible.
kw = dict(fps=fps, min_bout_s=None, bridge_gap_s=None)
lab_inst = bc.append_behavior_labels(df, velocity_column="ear_velocity_mm_s", **kw)
lab_smooth = bc.append_behavior_labels(df, velocity_column="ear_velocity_smooth_mm_s", **kw)

print(f"instantaneous velocity -> {n_movement_bouts(lab_inst['behavior'])} movement bouts")
print(f"smoothed velocity      -> {n_movement_bouts(lab_smooth['behavior'])} movement bouts")
print("\nSmoothing removes spike-driven spurious bouts at the source; the min-bout")
print("filter (default, used below) then removes any remaining short bouts.")
print("All classification below uses the smoothed velocity (the package default).")

## The threshold: raw vs log-scale Otsu

The speed histogram is dominated by a rest peak near zero with a long movement tail. Plain Otsu lands
far out on the tail (misses slow movement); log-scale Otsu sits just above the rest floor.

In [ ]:
speed = np.abs(df["ear_velocity_smooth_mm_s"].to_numpy())
thr_raw = bc.otsu_threshold(speed)
thr_log = bc.otsu_threshold(speed, log=True)
print(f"Otsu raw:  {thr_raw:.1f} mm/s   ({(speed >= thr_raw).mean():.1%} of frames above)")
print(f"Otsu log:  {thr_log:.1f} mm/s   ({(speed >= thr_log).mean():.1%} of frames above)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log1p(speed), bins=200, color="0.7")
ax.axvline(np.log1p(thr_raw), color="tab:red", ls="--", label=f"raw Otsu = {thr_raw:.0f} mm/s")
ax.axvline(np.log1p(thr_log), color="tab:green", ls="--", label=f"log Otsu = {thr_log:.0f} mm/s")
ax.set(xlabel="log1p(speed)", ylabel="count",
       title="Speed distribution (log scale) with Otsu thresholds")
ax.legend()
plt.tight_layout()
plt.show()

## Classify (defaults: smoothed velocity, log Otsu, 0.5 s minimum bout)

In [ ]:
out = bc.append_behavior_labels(df)  # all defaults
thr = out.attrs["behavior_velocity_threshold"]
counts = out["behavior"].value_counts()
print(f"threshold used: {thr:.1f} mm/s")
print(counts)
print(f"fraction moving: {(out['behavior'] == 'movement').mean():.3f}")

## Effect of the minimum bout duration

Raising `min_bout_s` removes progressively longer spurious bouts (fewer, longer movement bouts).

In [ ]:
rows = []
for mbs in [0.0, 0.1, 0.25, 0.5, 1.0]:
    lab = bc.append_behavior_labels(df, fps=fps, min_bout_s=(mbs or None))
    rows.append({"min_bout_s": mbs,
                 "movement_bouts": n_movement_bouts(lab["behavior"]),
                 "fraction_moving": (lab["behavior"] == "movement").mean()})
pd.DataFrame(rows)

## Ethogram: smoothed speed with movement shaded

In [ ]:
t = out["harp_time"].to_numpy()
t0 = t - t[0]
is_move = (out["behavior"] == "movement").to_numpy()

# a 60 s window
sl = slice(0, int(60 * fps))
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t0[sl], speed[sl], color="0.4", lw=0.7, label="smoothed speed")
ax.axhline(thr, color="tab:green", ls="--", lw=1, label=f"threshold ({thr:.0f} mm/s)")
ax.fill_between(t0[sl], 0, speed[sl].max(), where=is_move[sl], color="tab:orange",
                alpha=0.25, step="mid", label="movement")
ax.set(xlabel="time (s)", ylabel="speed (mm/s)", title="Rest / movement classification (60 s)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Bout-duration distributions

In [ ]:
def bout_durations_s(labels, value, fps):
    b = (np.asarray(labels) == value).astype(int)
    change = np.flatnonzero(np.diff(b)) + 1
    segs = np.split(b, change)
    return np.array([len(s) / fps for s in segs if s[0] == 1])


move_dur = bout_durations_s(out["behavior"], "movement", fps)
rest_dur = bout_durations_s(out["behavior"], "rest", fps)
print(f"movement bouts: {len(move_dur)}  (min {move_dur.min():.2f} s, median {np.median(move_dur):.2f} s)")
print(f"rest bouts:     {len(rest_dur)}  (median {np.median(rest_dur):.2f} s)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(move_dur, bins=np.logspace(np.log10(0.4), np.log10(max(move_dur.max(), 1)), 40),
        color="tab:orange", alpha=0.8, label="movement")
ax.axvline(0.5, color="k", ls="--", lw=1, label="min_bout_s = 0.5")
ax.set_xscale("log")
ax.set(xlabel="bout duration (s, log)", ylabel="count", title="Movement bout durations")
ax.legend()
plt.tight_layout()
plt.show()

The shortest movement bout is exactly 0.5 s — the minimum-bout filter guarantees every movement bout
is continuous for at least that long, while the sensitive log-Otsu threshold still admits slow
movement.